# 🫀 Heart Disease Prediction - Full Machine Learning Pipeline
This notebook contains the complete pipeline for the Heart Disease Prediction project, including Data Preprocessing, Model Training, Advanced Evaluation, **Hyperparameter Tuning (Optuna)**, Stacking Ensemble Optimization, Learning Curves, Explainable AI (SHAP & PDP), Feature Importance Comparison, and Data Export for Power BI.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
import time
import joblib
import shap
import os
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, precision_score, recall_score, roc_curve, precision_recall_curve, auc, accuracy_score, average_precision_score, ConfusionMatrixDisplay
from sklearn.cluster import KMeans
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTE

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier

os.makedirs('outputs', exist_ok=True)
os.makedirs('outputs/v2', exist_ok=True)

C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Data

In [2]:
# Load dataset from Excel
df = pd.read_excel('data/heart_disease_project_full.xlsx', sheet_name='Main_Dataset')
df_ml = df.copy()
display(df_ml.head())

,PatientID,HeartDisease,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,...,State_HD_Prevalence,State_Obesity_Rate,State_Smoking_Rate,State_Uninsured_Rate,State_Median_Income_K,State_Health_Rank,State_Health_Tier,State_HD_vs_National,State_Risk_Label,State_Population_M
0,P000001,No,16.60,Yes,No,No,3,30,No,Female,...,7.3,33.1,17.0,5.5,61,15,Above Average,-1.2,Below Average,3.19
1,P000002,No,20.34,No,No,Yes,0,0,No,Female,...,7.3,29.7,15.1,8.3,74,28,Below Average,-1.2,Below Average,8.63
2,P000003,No,26.58,Yes,No,No,20,30,No,Male,...,6.8,28.3,14.4,7.5,65,11,Above Average,-1.7,Below Average,4.24
3,P000004,No,24.21,No,No,No,0,0,No,Female,...,6.9,27.4,13.7,6.0,72,13,Above Average,-1.6,Below Average,20.20
4,P000005,No,23.71,No,No,No,28,0,Yes,Female,...,6.1,25.1,10.1,7.0,78,5,Top Tier,-2.4,Below Average,39.54


## 3. Data Cleaning and Preprocessing

In [3]:
# Drop redundant columns
drop_cols = [
    'PatientID', 'State_Code', 'State', 'AgeCategory', 'Diabetic',
    'Survey_Year', 'Survey_Quarter', 'Survey_Month',
    'State_HD_vs_National', 'State_Risk_Label', 'State_Health_Tier',
    'State_Population_M'
]
existing_drops = [c for c in drop_cols if c in df_ml.columns]
df_ml = df_ml.drop(columns=existing_drops)

# Binary Encoding (Yes/No -> 1/0)
binary_cols = ['HeartDisease', 'Smoking', 'AlcoholDrinking', 'Stroke', 'DiffWalking', 'PhysicalActivity', 'Diabetic_Binary', 'Asthma', 'KidneyDisease', 'SkinCancer']
for col in binary_cols:
    if col in df_ml.columns:
        df_ml[col] = df_ml[col].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

if 'Sex' in df_ml.columns:
    df_ml['Sex'] = df_ml['Sex'].map({'Male': 1, 'Female': 0}).fillna(0).astype(int)

# Ordinal Encoding
ordinal_maps = {
    'GenHealth': {'Poor': 0, 'Fair': 1, 'Good': 2, 'Very good': 3, 'Excellent': 4},
    'BMI_Category': {'Underweight': 0, 'Normal': 1, 'Overweight': 2, 'Obese': 3, 'Obese III': 3, 'Obese II': 3, 'Obese I': 3},
    'BMI_Risk': {'Low': 0, 'Medium': 1, 'High': 2, 'Very High': 2},
    'Comorbidity_Level': {'None': 0, 'Low': 1, 'Moderate': 2, 'High': 3},
    'Risk_Tier': {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3},
    'Lifestyle_Category': {'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3},
    'Sleep_Quality': {'Short': 0, 'Optimal': 1, 'Long': 2, 'Excessive': 2},
    'Age_Group': {'Young (18-34)': 0, 'Middle (35-54)': 1, 'Senior (55-69)': 2, 'Elderly (70+)': 3}
}
for col, mapping in ordinal_maps.items():
    if col in df_ml.columns:
        df_ml[col] = df_ml[col].map(mapping)
        median_val = df_ml[col].median()
        df_ml[col] = df_ml[col].fillna(median_val if not pd.isna(median_val) else 0).astype(int)

# One-Hot Encoding for Nominal variables
ohe_cols = [c for c in ['Race', 'Region'] if c in df_ml.columns]
if ohe_cols:
    df_ml = pd.get_dummies(df_ml, columns=ohe_cols, drop_first=True)

df_ml.columns = [str(c) for c in df_ml.columns]

## 4. Feature Engineering

In [4]:
# Create Interaction Features
interactions = {
    'Age_BMI_Interaction': ('Age_Numeric', 'BMI'),
    'Smoke_Diabetes': ('Smoking', 'Diabetic_Binary'),
    'Comorbidity_Age': ('Comorbidity_Count', 'Age_Numeric'),
    'Stroke_Kidney': ('Stroke', 'KidneyDisease'),
}

for new_col, (a, b) in interactions.items():
    if a in df_ml.columns and b in df_ml.columns:
        df_ml[new_col] = df_ml[a] * df_ml[b]

## 5. Train-Test Split and Scaling

In [5]:
# Split Target and Features
X = df_ml.drop(['HeartDisease', 'BMI', 'BMI_Risk', 'BMI_Category', 'Age_BMI_Interaction'], axis=1, errors='ignore')
y = df_ml['HeartDisease']

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling Numerical Columns
num_cols = ['BMI', 'Age_Numeric', 'PhysicalHealth', 'MentalHealth', 
            'SleepTime', 'Risk_Score', 'Lifestyle_Score', 'Health_Score',
            'Comorbidity_Count', 'State_HD_Prevalence', 'State_Obesity_Rate',
            'State_Smoking_Rate', 'State_Uninsured_Rate', 
            'State_Median_Income_K', 'State_Health_Rank',
            'Age_BMI_Interaction', 'Comorbidity_Age']
num_cols = [c for c in num_cols if c in X_train.columns]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

## 6. Handling Imbalanced Data (SMOTE)

In [6]:
# Apply SMOTE only on training data to balance the classes
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Original class distribution:\n", y_train.value_counts())
print("\nSMOTE class distribution:\n", y_train_sm.value_counts())

Original class distribution:
 HeartDisease
0    233938
1     21898
Name: count, dtype: int64

SMOTE class distribution:
 HeartDisease
0    233938
1    233938
Name: count, dtype: int64


## 7. Model Training (V1 - Baseline)
We train the 6 base models with manually selected parameters.

In [7]:
# 1. Logistic Regression
lr = LogisticRegression(C=1.0, solver='lbfgs', class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=10, min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# 3. XGBoost
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=10.7, eval_metric='auc', random_state=42)
xgb.fit(X_train, y_train)

# 4. LightGBM
lgbm = LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.05, is_unbalance=True, random_state=42, n_jobs=-1, verbose=-1)
lgbm.fit(X_train, y_train)

# 5. Deep Learning (Neural Network)
nn = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu', solver='adam', alpha=0.001,
    batch_size=256, learning_rate='adaptive',
    learning_rate_init=0.001, max_iter=100,
    early_stopping=True, validation_fraction=0.1,
    random_state=42
)
nn.fit(X_train_sm, y_train_sm)

# 6. Stacking Ensemble
stacking = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(C=1.0, solver='lbfgs', class_weight='balanced', max_iter=1000, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1)),
        ('xgb', XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, scale_pos_weight=10.7, eval_metric='auc', random_state=42)),
    ],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=3, n_jobs=-1
)
stacking.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('lr', ...), ('rf', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegre...max_iter=1000)
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",3
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",-1
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0I

## 8. Model Evaluation (V1)
We evaluate V1 baseline models on the held-out test set.

In [8]:
def evaluate_model(model, X_test, y_test, model_name: str, threshold: float = 0.5):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    roc_auc  = roc_auc_score(y_test, y_proba)
    pr_auc   = average_precision_score(y_test, y_proba)
    f1       = f1_score(y_test, y_pred)
    recall   = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    acc      = accuracy_score(y_test, y_pred)

    metrics = {
        'Model': model_name,
        'ROC-AUC': round(roc_auc, 4),
        'PR-AUC': round(pr_auc, 4),
        'F1-Score': round(f1, 4),
        'Recall': round(recall, 4),
        'Precision': round(precision, 4),
        'Accuracy': round(acc, 4),
    }
    return metrics, y_proba, y_pred

models_v1 = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'XGBoost': xgb,
    'LightGBM': lgbm,
    'Neural Network': nn,
    'Stacking Ensemble': stacking
}

results_v1 = []
for name, model in models_v1.items():
    mets, _, _ = evaluate_model(model, X_test, y_test, name)
    results_v1.append(mets)

results_v1_df = pd.DataFrame(results_v1).sort_values('ROC-AUC', ascending=False)
display(results_v1_df)

,Model,ROC-AUC,PR-AUC,F1-Score,Recall,Precision,Accuracy
5,Stacking Ensemble,0.8393,0.3513,0.3444,0.7885,0.2203,0.7430
0,Logistic Regression,0.8374,0.3491,0.3447,0.7848,0.2208,0.7445
3,LightGBM,0.8374,0.3480,0.3441,0.7823,0.2205,0.7447
2,XGBoost,0.8341,0.3407,0.3454,0.7724,0.2224,0.7493
1,Random Forest,0.8312,0.3312,0.3627,0.6789,0.2475,0.7958
4,Neural Network,0.7806,0.2305,0.2708,0.2738,0.2678,0.8738


## 9. Hyperparameter Tuning using Optuna (V2)
We tune hyperparameters using Optuna (Bayesian Optimization) on a stratified subset of 50,000 samples for efficiency.

In [9]:
# Stratified sample for faster tuning
np.random.seed(42)
n_sample = min(50000, X_train.shape[0])
sample_idx = np.random.choice(X_train.shape[0], n_sample, replace=False)
X_tune = X_train.iloc[sample_idx]
y_tune = y_train.iloc[sample_idx]

cv_folds = 3
cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# 1. XGBoost Objective
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'scale_pos_weight': pos_weight,
        'eval_metric': 'auc',
        'random_state': 42
    }
    model = XGBClassifier(**params)
    return cross_val_score(model, X_tune, y_tune, cv=cv, scoring='roc_auc', n_jobs=-1).mean()

# 2. LightGBM Objective
def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800, step=100),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'is_unbalance': True,
        'random_state': 42,
        'verbose': -1
    }
    model = LGBMClassifier(**params)
    return cross_val_score(model, X_tune, y_tune, cv=cv, scoring='roc_auc', n_jobs=-1).mean()

print("[Tuning Demostration] Study setup complete. Run Optuna optimize studies to find optimal parameters.")

[Tuning Demostration] Study setup complete. Run Optuna optimize studies to find optimal parameters.


## 10. Training Optimized V2 Models & Stacking Ensemble V2
We train final models using the tuned parameters found by Optuna on the full dataset.

In [10]:
# Using optimized hyperparameters from Optuna studies (already determined)
best_xgb_params = {
    'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.0225,
    'subsample': 0.758, 'colsample_bytree': 0.561, 'min_child_weight': 7,
    'gamma': 3.949, 'scale_pos_weight': pos_weight, 'eval_metric': 'auc', 'random_state': 42
}

best_lgbm_params = {
    'n_estimators': 400, 'num_leaves': 18, 'learning_rate': 0.0162,
    'min_child_samples': 35, 'subsample': 0.662, 'colsample_bytree': 0.546,
    'reg_alpha': 4.53, 'is_unbalance': True, 'random_state': 42, 'verbose': -1
}

best_rf_params = {
    'n_estimators': 500, 'max_depth': 7, 'min_samples_split': 4,
    'min_samples_leaf': 10, 'max_features': 0.7, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1
}

# Train tuned models
rf_v2 = RandomForestClassifier(**best_rf_params).fit(X_train, y_train)
xgb_v2 = XGBClassifier(**best_xgb_params).fit(X_train, y_train)
lgbm_v2 = LGBMClassifier(**best_lgbm_params).fit(X_train, y_train)

# Train Stacking Ensemble V2 (LR + Tuned RF + Tuned XGB + Tuned LGBM -> Logistic Regression)
stacking_v2 = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(C=1.0, solver='lbfgs', class_weight='balanced', max_iter=1000, random_state=42)),
        ('rf', RandomForestClassifier(**best_rf_params)),
        ('xgb', XGBClassifier(**best_xgb_params)),
        ('lgbm', LGBMClassifier(**best_lgbm_params))
    ],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=5, n_jobs=-1
)
stacking_v2.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('lr', ...), ('rf', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegre...max_iter=1000)
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",5
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",-1
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0I

## 11. Model Evaluation (V2)
Evaluating the optimized V2 models on the test set.

In [11]:
models_v2 = {
    'Random Forest V2': rf_v2,
    'XGBoost V2': xgb_v2,
    'LightGBM V2': lgbm_v2,
    'Stacking V2': stacking_v2
}

results_v2 = []
# Include LR and Neural Network V1 for completeness
results_v2.append(results_v1_df[results_v1_df['Model']=='Logistic Regression'].iloc[0].to_dict())
results_v2.append(results_v1_df[results_v1_df['Model']=='Neural Network'].iloc[0].to_dict())
results_v2.rename(columns={'Neural Network': 'Neural Network V2'}, inplace=True) if isinstance(results_v2, pd.DataFrame) else None

for name, model in models_v2.items():
    mets, _, _ = evaluate_model(model, X_test, y_test, name)
    results_v2.append(mets)

results_v2_df = pd.DataFrame(results_v2).sort_values('ROC-AUC', ascending=False)
display(results_v2_df)

,Model,ROC-AUC,PR-AUC,F1-Score,Recall,Precision,Accuracy
4,LightGBM V2,0.8405,0.3526,0.3395,0.8044,0.2151,0.7321
5,Stacking V2,0.8403,0.3532,0.3442,0.7927,0.2198,0.7414
3,XGBoost V2,0.8397,0.3523,0.3397,0.8020,0.2154,0.7330
0,Logistic Regression,0.8374,0.3491,0.3447,0.7848,0.2208,0.7445
2,Random Forest V2,0.8363,0.3409,0.3332,0.8049,0.2101,0.7242
1,Neural Network,0.7806,0.2305,0.2708,0.2738,0.2678,0.8738


## 12. Threshold Tuning (Youden's J Statistic) for Best V2 Model
Finding the optimal classification threshold for our best V2 model (LightGBM V2).

In [12]:
best_model_v2 = lgbm_v2
y_proba_best = best_model_v2.predict_proba(X_test)[:, 1]
fpr_b, tpr_b, thresholds_b = roc_curve(y_test, y_proba_best)

# Youden's J statistic
j_scores = tpr_b - fpr_b
best_idx = np.argmax(j_scores)
optimal_threshold = thresholds_b[best_idx]

print(f"Optimal Threshold for V2: {optimal_threshold:.4f}")

# Re-evaluate with new threshold
_, _, y_pred_tuned = evaluate_model(best_model_v2, X_test, y_test, 'LightGBM V2 (Tuned)', threshold=optimal_threshold)
tuned_metrics, _, _ = evaluate_model(best_model_v2, X_test, y_test, 'LightGBM V2 (Tuned)', threshold=optimal_threshold)
display(pd.DataFrame([tuned_metrics]))

Optimal Threshold for V2: 0.4750


,Model,ROC-AUC,PR-AUC,F1-Score,Recall,Precision,Accuracy
0,LightGBM V2 (Tuned),0.8405,0.3526,0.3319,0.8289,0.2075,0.7144


## 13. V1 vs V2 Model Comparison
We compare performance between V1 (manual hyperparameters) and V2 (Optuna optimized hyperparameters).

In [13]:
# Compare best models
v1_best = results_v1_df.iloc[0]
v2_best = results_v2_df.iloc[0]

print(f"★ V1 Best Model ({v1_best['Model']}) ROC-AUC: {v1_best['ROC-AUC']:.4f}")
print(f"★ V2 Best Model ({v2_best['Model']}) ROC-AUC: {v2_best['ROC-AUC']:.4f}")
print(f"★ Absolute AUC Improvement: {v2_best['ROC-AUC'] - v1_best['ROC-AUC']:+.4f}")

# Plot bar chart using Plotly
metrics_to_compare = ['ROC-AUC', 'PR-AUC', 'F1-Score', 'Recall', 'Precision', 'Accuracy']
v1_vals = [v1_best[m] for m in metrics_to_compare]
v2_vals = [v2_best[m] for m in metrics_to_compare]

fig = go.Figure()
fig.add_trace(go.Bar(
    name=f"V1: {v1_best['Model']}",
    x=metrics_to_compare,
    y=v1_vals,
    text=[f"{v:.3f}" for v in v1_vals],
    textposition='outside',
    marker_color='#90CAF9',
    marker_line=dict(color='#1565C0', width=1.5),
))
fig.add_trace(go.Bar(
    name=f"V2: {v2_best['Model']}",
    x=metrics_to_compare,
    y=v2_vals,
    text=[f"{v:.3f}" for v in v2_vals],
    textposition='outside',
    marker_color='#EF9A9A',
    marker_line=dict(color='#C62828', width=1.5),
))

fig.update_layout(
    barmode='group',
    title='V1 Baseline vs V2 Optimized — Best Model Comparison',
    yaxis=dict(title='Score', range=[0, 1.15]),
    xaxis=dict(title='Metric'),
    template='plotly_dark',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
)
fig.show()

★ V1 Best Model (Stacking Ensemble) ROC-AUC: 0.8393
★ V2 Best Model (LightGBM V2) ROC-AUC: 0.8405
★ Absolute AUC Improvement: +0.0012


## 14. Learning Curves
Learning curves help diagnose if the tuned model is suffering from high bias (underfitting) or high variance (overfitting).

In [14]:
# Generate learning curve on XGBoost V2 for speed
train_sizes, train_scores, test_scores = learning_curve(
    xgb_v2, X_train, y_train, cv=3, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 6), n_jobs=-1, random_state=42
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

fig = go.Figure()

# Training score band
fig.add_trace(go.Scatter(
    x=np.concatenate([train_sizes, train_sizes[::-1]]),
    y=np.concatenate([train_mean + train_std, (train_mean - train_std)[::-1]]),
    fill='toself',
    fillcolor='rgba(33,150,243,0.1)',
    line=dict(color='rgba(0,0,0,0)'),
    showlegend=False,
    name='Training ±1 std',
))
# Training score line
fig.add_trace(go.Scatter(
    x=train_sizes, y=train_mean,
    mode='lines+markers',
    name='Training Score',
    line=dict(color='#2196F3', width=2),
    marker=dict(size=8),
))

# CV score band
fig.add_trace(go.Scatter(
    x=np.concatenate([train_sizes, train_sizes[::-1]]),
    y=np.concatenate([test_mean + test_std, (test_mean - test_std)[::-1]]),
    fill='toself',
    fillcolor='rgba(244,67,54,0.1)',
    line=dict(color='rgba(0,0,0,0)'),
    showlegend=False,
    name='CV ±1 std',
))
# CV score line
fig.add_trace(go.Scatter(
    x=train_sizes, y=test_mean,
    mode='lines+markers',
    name='Cross-Validation Score',
    line=dict(color='#F44336', width=2),
    marker=dict(size=8),
))

fig.update_layout(
    title='Learning Curve — XGBoost V2 (Tuned)',
    xaxis=dict(title='Training Set Size', gridcolor='rgba(255,255,255,0.1)'),
    yaxis=dict(title='ROC-AUC Score', gridcolor='rgba(255,255,255,0.1)'),
    template='plotly_dark',
    height=500,
    legend=dict(x=0.7, y=0.05),
)
fig.show()

## 15. Explainable AI (SHAP)
Using SHAP values on the XGBoost V2 model to interpret global and local feature contributions.

In [16]:
# SHAP Explanations on Tuned XGBoost V2
explainer = shap.TreeExplainer(xgb_v2)

# Sample 1000 rows for faster SHAP calculation
X_sample = X_test.sample(1000, random_state=42)
shap_values = explainer.shap_values(X_sample)

# 1. SHAP Global Importance (Bar) — Plotly
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_imp_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance': mean_abs_shap
}).sort_values('importance', ascending=True).tail(20)

fig1 = go.Figure(go.Bar(
    x=shap_imp_df['importance'],
    y=shap_imp_df['feature'],
    orientation='h',
    marker_color=px.colors.sequential.Reds_r[:len(shap_imp_df)],
    text=[f"{v:.4f}" for v in shap_imp_df['importance']],
    textposition='outside',
))
fig1.update_layout(
    title='SHAP Global Feature Importance (mean |SHAP value|)',
    xaxis=dict(title='Mean |SHAP Value|', gridcolor='rgba(255,255,255,0.05)'),
    yaxis=dict(title=''),
    template='plotly_dark',
    height=600,
    margin=dict(l=200, r=60, t=50, b=40),
)
fig1.show()

# 2. SHAP Beeswarm Plot — Plotly Scatter
# For each feature, plot SHAP value vs jittered y position, colored by feature value
top_features = shap_imp_df['feature'].tolist()[-15:]  # Top 15
bee_data = []
for i, feat in enumerate(top_features):
    col_idx = list(X_test.columns).index(feat)
    sv = shap_values[:, col_idx]
    fv = X_sample[feat].values.astype(float)
    # Normalize feature values for color
    fv_min, fv_max = fv.min(), fv.max()
    fv_norm = (fv - fv_min) / (fv_max - fv_min + 1e-10)
    jitter = np.random.normal(0, 0.15, len(sv))
    for s, f_n, j in zip(sv, fv_norm, jitter):
        bee_data.append({'feature': feat, 'shap_value': s, 'feature_value_norm': f_n, 'y_jitter': i + j})

bee_df = pd.DataFrame(bee_data)

fig2 = go.Figure(go.Scatter(
    x=bee_df['shap_value'],
    y=bee_df['y_jitter'],
    mode='markers',
    marker=dict(
        color=bee_df['feature_value_norm'],
        colorscale='RdBu_r',
        size=3,
        opacity=0.5,
        colorbar=dict(title='Feature Value<br>(normalized)', thickness=15),
    ),
    text=bee_df['feature'],
    hovertemplate='<b>%{text}</b><br>SHAP: %{x:.4f}<extra></extra>',
))
fig2.update_layout(
    title='SHAP Beeswarm Plot (Top 15 Features)',
    xaxis=dict(title='SHAP Value', zeroline=True, zerolinecolor='rgba(255,255,255,0.3)', gridcolor='rgba(255,255,255,0.05)'),
    yaxis=dict(
        tickmode='array',
        tickvals=list(range(len(top_features))),
        ticktext=top_features,
        gridcolor='rgba(255,255,255,0.05)',
    ),
    template='plotly_dark',
    height=600,
    margin=dict(l=200, r=60, t=50, b=40),
)
fig2.show()

In [ ]:
# 3. SHAP Waterfall (For a high-risk patient) — Plotly
y_pred_proba = xgb_v2.predict_proba(X_sample)[:, 1]
high_risk_idx = np.argmax(y_pred_proba)

sv = shap_values[high_risk_idx]
feat_names = X_test.columns.tolist()

# Get top 15 features by absolute SHAP value
wf_df = pd.DataFrame({
    'feature': feat_names,
    'shap_value': sv,
    'abs_shap': np.abs(sv)
}).sort_values('abs_shap', ascending=False).head(15)
wf_df = wf_df.sort_values('shap_value', ascending=True)

colors = ['#E74C3C' if v > 0 else '#3498DB' for v in wf_df['shap_value']]

fig3 = go.Figure(go.Bar(
    x=wf_df['shap_value'],
    y=wf_df['feature'],
    orientation='h',
    marker_color=colors,
    text=[f"{v:+.3f}" for v in wf_df['shap_value']],
    textposition='outside',
    textfont=dict(size=11),
))
fig3.update_layout(
    title=f'SHAP Waterfall — High-Risk Patient (Predicted Prob: {y_pred_proba[high_risk_idx]:.2%})',
    xaxis=dict(
        title='SHAP Value (impact on prediction)',
        zeroline=True, zerolinecolor='rgba(255,255,255,0.3)', zerolinewidth=2,
        gridcolor='rgba(255,255,255,0.05)',
    ),
    yaxis=dict(title='', gridcolor='rgba(255,255,255,0.05)'),
    template='plotly_dark',
    height=500,
    margin=dict(l=200, r=80, t=60, b=50),
)
fig3.show()

print(f"\n📐 Base value = {explainer.expected_value:.3f}")
print(f"   Final prediction = {explainer.expected_value + sv.sum():.3f}")
print(f"   Total SHAP shift = {sv.sum():+.3f}")


📐 Base value = 0.003
   Final prediction = 3.163
   Total SHAP shift = +3.161


In [ ]:
# 4. SHAP Dependence Plots — Plotly
def plotly_shap_dependence(feature, shap_values, X_data, feature_names):
    """Create a Plotly SHAP dependence plot."""
    col_idx = list(feature_names).index(feature)
    sv = shap_values[:, col_idx]
    fv = X_data[feature].values
    
    # Find the best interaction feature (highest correlation with SHAP values)
    correlations = []
    for i, fname in enumerate(feature_names):
        if fname == feature:
            correlations.append(0)
            continue
        corr = np.abs(np.corrcoef(X_data.iloc[:, i].values, sv)[0, 1])
        correlations.append(corr if not np.isnan(corr) else 0)
    interact_idx = np.argmax(correlations)
    interact_vals = X_data.iloc[:, interact_idx].values
    interact_name = feature_names[interact_idx]
    
    fig = go.Figure(go.Scatter(
        x=fv,
        y=sv,
        mode='markers',
        marker=dict(
            color=interact_vals,
            colorscale='RdBu_r',
            size=4,
            opacity=0.6,
            colorbar=dict(title=interact_name, thickness=15),
        ),
        hovertemplate=f'<b>{feature}</b>: %{{x:.2f}}<br>SHAP: %{{y:.4f}}<extra></extra>',
    ))
    fig.update_layout(
        title=f'SHAP Dependence Plot: {feature}',
        xaxis=dict(title=feature, gridcolor='rgba(255,255,255,0.05)'),
        yaxis=dict(title=f'SHAP value for {feature}', zeroline=True,
                   zerolinecolor='rgba(255,255,255,0.3)', gridcolor='rgba(255,255,255,0.05)'),
        template='plotly_dark',
        height=450,
    )
    return fig

feat_names_list = X_test.columns.tolist()
if 'Age_BMI_Interaction' in X_test.columns:
    plotly_shap_dependence('Age_BMI_Interaction', shap_values, X_sample, feat_names_list).show()
if 'Age_Numeric' in X_test.columns:
    plotly_shap_dependence('Age_Numeric', shap_values, X_sample, feat_names_list).show()

## 16. Feature Importance Comparison
Comparing feature importance across Gini (Random Forest V2), Permutation (Stacking V2), and SHAP (XGBoost V2).

In [ ]:
feature_names = X_test.columns.tolist()

# 1. Gini Importance (RF V2)
feat_imp_gini = pd.Series(rf_v2.feature_importances_, index=feature_names).sort_values(ascending=False)

# 2. Permutation Importance (Stacking V2) on a small subset to save time
X_perm = X_test.sample(200, random_state=42)
y_perm = y_test.loc[X_perm.index]
perm_imp = permutation_importance(stacking_v2, X_perm, y_perm, n_repeats=3, random_state=42, n_jobs=-1)
feat_imp_perm = pd.Series(perm_imp.importances_mean, index=feature_names).sort_values(ascending=False)

# 3. SHAP Importance
feat_imp_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names).sort_values(ascending=False)

imp_df = pd.DataFrame({
    'Gini (RF V2)': feat_imp_gini.head(10),
    'Permutation (Stacking V2)': feat_imp_perm.head(10),
    'SHAP Mean |val| (XGB V2)': feat_imp_shap.head(10)
})
display(imp_df)

# Plotly visualization — Top 10 Features by each method
top_n = 10
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=['Gini Importance (RF V2)',
                                   'Permutation Importance (Stacking V2)',
                                   'SHAP Mean |val| (XGB V2)'],
                    horizontal_spacing=0.45)

for col_num, (series, color) in enumerate([
    (feat_imp_gini.head(top_n).sort_values(), '#2196F3'),
    (feat_imp_perm.head(top_n).sort_values(), '#FF9800'),
    (feat_imp_shap.head(top_n).sort_values(), '#E74C3C'),
], start=1):
    fig.add_trace(go.Bar(
        x=series.values,
        y=series.index,
        orientation='h',
        marker_color=color,
        text=[f"{v:.4f}" for v in series.values],
        textposition='inside',
        showlegend=False,
    ), row=1, col=col_num)

fig.update_layout(
    title='Feature Importance Comparison (Top 10, width=1500)',
    template='plotly_dark',
    height=500,
    margin=dict(l=160, r=40, t=60, b=40),
)
fig.show()

,Gini (RF V2),Permutation (Stacking V2),SHAP Mean |val| (XGB V2)
Age_BMI_Interaction,0.064634,NaN,0.153158
Age_Group,0.005419,NaN,NaN
Age_Numeric,0.155233,NaN,0.552730
BMI,NaN,0.003333,NaN
Comorbidity_Age,0.384889,NaN,0.290498
GenHealth,0.060593,NaN,0.170628
Health_Score,0.083850,NaN,0.286175
MentalHealth,NaN,0.005000,NaN
PhysicalHealth,NaN,0.001667,0.045098
Race_Black,NaN,0.005000,NaN


## 17. Partial Dependence Plots (PDP)

In [ ]:
# Partial Dependence Plots (PDP) — Plotly
features_for_pdp = ['Age_BMI_Interaction', 'Age_Numeric', 'Comorbidity_Count']
features_for_pdp = [f for f in features_for_pdp if f in X_test.columns]

if features_for_pdp:
    from sklearn.inspection import partial_dependence
    
    fig = make_subplots(
        rows=1, cols=len(features_for_pdp),
        subplot_titles=[f'PDP: {f}' for f in features_for_pdp],
                    horizontal_spacing=0.45)
    )
    
    colors = ['#2196F3', '#E74C3C', '#4CAF50']
    
    for i, feat in enumerate(features_for_pdp):
        pd_result = partial_dependence(xgb_v2, X_sample, features=[feat], grid_resolution=30)
        values = pd_result['grid_values'][0]
        avg_pred = pd_result['average'][0]
        
        # Confidence band (using individual predictions if available)
        try:
            pd_ind = partial_dependence(xgb_v2, X_sample, features=[feat], grid_resolution=30, kind='individual')
            ind_preds = pd_ind['individual'][0]
            std = ind_preds.std(axis=0)
            
            fig.add_trace(go.Scatter(
                x=np.concatenate([values, values[::-1]]),
                y=np.concatenate([avg_pred + std, (avg_pred - std)[::-1]]),
                fill='toself',
                fillcolor=f'rgba({int(colors[i][1:3], 16)},{int(colors[i][3:5], 16)},{int(colors[i][5:7], 16)},0.1)',
                line=dict(color='rgba(0,0,0,0)'),
                showlegend=False,
            ), row=1, col=i+1)
        except Exception:
            pass
        
        fig.add_trace(go.Scatter(
            x=values,
            y=avg_pred,
            mode='lines',
            name=feat,
            line=dict(color=colors[i], width=2.5),
            showlegend=False,
        ), row=1, col=i+1)
        
        fig.update_xaxes(title_text=feat, row=1, col=i+1, gridcolor='rgba(255,255,255,0.05)')
        fig.update_yaxes(title_text='Partial Dependence' if i == 0 else '', row=1, col=i+1, gridcolor='rgba(255,255,255,0.05)')
    
    fig.update_layout(
        title='Partial Dependence Plots (XGBoost V2)',
        template='plotly_dark',
        height=450,
        margin=dict(t=80, b=50),
    )
    fig.show()

## 18. K-Means Patient Segmentation & Export for Power BI
We enrich the original dataset with patient risks from our optimized model, run K-Means patient segmentation, and export the resulting data to Excel.

In [ ]:
# Predict on the entire dataset using the best tuned model (LightGBM V2)
X_all = df_ml.drop('HeartDisease', axis=1)
X_all_scaled = X_all.copy()
X_all_scaled[num_cols] = scaler.transform(X_all_scaled[num_cols])

df['ML_Probability'] = best_model_v2.predict_proba(X_all_scaled)[:, 1]
df['ML_Prediction'] = (df['ML_Probability'] >= optimal_threshold).astype(int)

# K-Means Clustering
cluster_features = ['BMI', 'Age_Numeric', 'Risk_Score', 'Comorbidity_Count', 'Lifestyle_Score', 'Health_Score']
X_cluster = StandardScaler().fit_transform(df[cluster_features])
kmeans = KMeans(n_clusters=5, random_state=42)
df['Patient_Segment'] = kmeans.fit_predict(X_cluster)

# Export to Excel for Power BI
export_path = 'outputs/heart_disease_with_predictions.xlsx'
df.to_excel(export_path, index=False)
print(f"Dataset successfully enriched and exported to {export_path}")

Dataset successfully enriched and exported to outputs/heart_disease_with_predictions.xlsx
